In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs

import rasterio
import numpy as np
import os

In [2]:


def generate_landusef_from_lu_index(lu_index, nclass=None, dtype=np.float32):
    """
    Generate a LANDUSEF-style array from a LU_INDEX array.

    Parameters:
        lu_index (np.ndarray): 2D array of land use class indices (ny, nx).
        nclass (int): Number of land use classes.
        dtype (np.dtype): Data type for the output array.

    Returns:
        np.ndarray: Array of shape (1, nclass, ny, nx) with one-hot encoding for each class.
    """
    # Ensure lu_index is 2D (ny, nx)
    if lu_index.ndim == 3 and lu_index.shape[0] == 1:
        lu_index = lu_index[0]
    elif lu_index.ndim > 2:
        raise ValueError(f"Unexpected lu_index shape: {lu_index.shape}")

    ny, nx = lu_index.shape
    landuse_generated = np.zeros((1, nclass, ny, nx), dtype=dtype)

    for class_idx in range(nclass):
        # Class numbers are assumed to be 1-based (i.e., class 1 is index 0)
        landuse_generated[0, class_idx, :, :] = (lu_index == (class_idx + 1)).astype(dtype)

    return landuse_generated



In [3]:
import matplotlib.colors as mcolors
import numpy as np

# Define the 40-class LULC mapping as a dictionary: {class_number: {"label": ..., "color": ...}}
lulc_dict = {
    1:  {"label": "Evergreen Needleleaf Forest",         "color": "#05450a"},
    2:  {"label": "Evergreen Broadleaf Forest",          "color": "#086a10"},
    3:  {"label": "Deciduous Needleleaf Forest",         "color": "#54a708"},
    4:  {"label": "Deciduous Broadleaf Forest",          "color": "#78d203"},
    5:  {"label": "Mixed Forests",                       "color": "#009900"},
    6:  {"label": "Closed Shrublands",                   "color": "#c6b044"},
    7:  {"label": "Open Shrublands",                     "color": "#dcd159"},
    8:  {"label": "Woody Savannas",                      "color": "#dade48"},
    9:  {"label": "Savannas",                            "color": "#fbff13"},
    10: {"label": "Grasslands",                          "color": "#b6ff05"},
    11: {"label": "Permanent Wetlands",                  "color": "#27ff87"},
    12: {"label": "Croplands",                           "color": "#006400"},
    13: {"label": "Urban and Built Up",                  "color": "#FF0000"},
    14: {"label": "Cropland/Natural Vegetation Mosaic",  "color": "#ADFF2F"},
    15: {"label": "Permanent Snow and Ice",              "color": "#69fff8"},
    16: {"label": "Barren or Sparsely Vegetated",        "color": "#f9ffa4"},
    17: {"label": "IGBP Water",                          "color": "#1c0dff"},
    18: {"label": "Unclassified",                        "color": "#cccccc"},
    19: {"label": "Fill Value",                          "color": "#999999"},
    20: {"label": "Unclassified",                        "color": "#cccccc"},
    21: {"label": "Open Water",                          "color": "#1c0dff"},
    22: {"label": "Perennial Ice/Snow",                  "color": "#69fff8"},
    23: {"label": "Developed Open Space",                "color": "#ffb3b3"},
    24: {"label": "Developed Low Intensity",             "color": "#ff6666"},
    25: {"label": "Developed Medium Intensity",          "color": "#cc0000"},
    26: {"label": "Developed High Intensity",            "color": "#990000"},
    27: {"label": "Barren Land (Rock/Sand/Clay)",        "color": "#e2e2e2"},
    28: {"label": "Deciduous Forest",                    "color": "#b2df8a"},
    29: {"label": "Evergreen Forest",                    "color": "#33a02c"},
    30: {"label": "Mixed Forest",                        "color": "#6a3d9a"},
    31: {"label": "Dwarf Scrub",                         "color": "#cab2d6"},
    32: {"label": "Shrub/Scrub",                         "color": "#ffff99"},
    33: {"label": "Grassland/Herbaceous",                "color": "#b15928"},
    34: {"label": "Sedge/Herbaceous",                    "color": "#8dd3c7"},
    35: {"label": "Lichens",                             "color": "#fb8072"},
    36: {"label": "Moss",                                "color": "#80b1d3"},
    37: {"label": "Pasture/Hay",                         "color": "#fdb462"},
    38: {"label": "Cultivated Crops",                    "color": "#ffd92f"},
    39: {"label": "Woody Wetlands",                      "color": "#a6cee3"},
    40: {"label": "Emergent Herbaceous Wetlands",        "color": "#1f78b4"},
}

# Extract color list and label list in order for plotting
lulc_colors = [lulc_dict[i]["color"] for i in range(1, 41)]
lulc_labels = [lulc_dict[i]["label"] for i in range(1, 41)]

cmap_40 = mcolors.ListedColormap(lulc_colors)


In [4]:
nlcd_raw_map_folder = '/nas/rstor/akumar/USA/PhD/Objective01/Hurricane_Harvey/update_geog/update_geog/NLCD/'

year = '2017'


infile_geog01 = '/nas/rstor/akumar/USA/PhD/Objective01/Hurricane_Harvey/WRF_Harvey_V2/WRF_Simulations/WRF_FNL_2512/pre/WPS_4.5/geo_em.d02_from_derecho.nc'


In [5]:
# --- Read original geo_em file and extract lat/lon and LU_INDEX ---


infile_geog01_xr = xr.open_dataset(infile_geog01)

geog_latitudes = infile_geog01_xr['XLAT_M'].values.squeeze()
geog_longitudes = infile_geog01_xr['XLONG_M'].values.squeeze()

geog_latitudes_1d = geog_latitudes[:, 0]
geog_longitudes_1d = geog_longitudes[0, :]

geog01_lulc = infile_geog01_xr['LU_INDEX'].values.squeeze()


geog_roi_domain = {'lon_min': geog_longitudes_1d[0], 'lon_max': geog_longitudes_1d[-1], 
                   'lat_min': geog_latitudes_1d[0], 'lat_max': geog_latitudes_1d[-1]}


margin = 1
nlcd_crop_domain = {'lon_min': geog_roi_domain['lon_min'] - margin, 'lon_max': geog_roi_domain['lon_max'] + margin, 
                    'lat_min': geog_roi_domain['lat_min'] - margin, 'lat_max': geog_roi_domain['lat_max'] + margin}

print(nlcd_crop_domain)
print(geog_roi_domain)


{'lon_min': -101.26945495605469, 'lon_max': -86.30169677734375, 'lat_min': 21.84516143798828, 'lat_max': 34.29186248779297}
{'lon_min': -100.269455, 'lon_max': -87.3017, 'lat_min': 22.845161, 'lat_max': 33.291862}


In [16]:
import subprocess
import os

def crop_nlcd_with_gdalwarp(raw_NLCD_map, year, nlcd_domain, nlcd_processed_tmp_folder='~/tmp/nlcd_processed/', output_NLCD_crop_filename=None, 
                            force_recalculate=False):
    """
    Crop the NLCD GeoTIFF using gdalwarp for a given year and domain.

    Parameters:
        raw_NLCD_map (str): Path to the raw NLCD GeoTIFF file.
        year (int or str): Year of the NLCD data (used in output filename).
        nlcd_domain (dict): Dictionary with keys 'lon_min', 'lon_max', 'lat_min', 'lat_max'.

    Returns:
        output_NLCD_crop_filename (str): Path to the cropped NLCD GeoTIFF.
    """
    import random
    import string

    # Expand user in output folder path and ensure directory exists
    nlcd_processed_tmp_folder_expanded = os.path.expanduser(nlcd_processed_tmp_folder)
    os.makedirs(nlcd_processed_tmp_folder_expanded, exist_ok=True)

    if output_NLCD_crop_filename is None:
        rand_prefix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
        output_NLCD_crop_filename = os.path.join(nlcd_processed_tmp_folder_expanded, f'{rand_prefix}_NLCD_{year}_crop.tif')
    else:
        output_NLCD_crop_filename = os.path.join(nlcd_processed_tmp_folder_expanded, output_NLCD_crop_filename)


    if os.path.exists(output_NLCD_crop_filename):
        print(f"Cropped NLCD file already exists: {output_NLCD_crop_filename}. Use force_recalculate=True to overwrite.")
        return output_NLCD_crop_filename

    te_args = [
        str(nlcd_domain['lon_min']),
        str(nlcd_domain['lat_min']),
        str(nlcd_domain['lon_max']),
        str(nlcd_domain['lat_max'])
    ]

    gdalwarp_cmd = [
        "gdalwarp",
        "-t_srs", "EPSG:4326",
        "-te"
    ] + te_args + [
        "-te_srs", "EPSG:4326",
        raw_NLCD_map,
        output_NLCD_crop_filename
    ]

    print("Running gdalwarp command:", " ".join(gdalwarp_cmd))
    subprocess.run(gdalwarp_cmd, check=True)
    return output_NLCD_crop_filename

In [17]:
raw_NLCD_map = f'{nlcd_raw_map_folder}/Annual_NLCD_LndCov_{year}_CU_C1V1.tif'
if not os.path.exists(raw_NLCD_map):
    try:
        raise FileNotFoundError(f"Raw NLCD map file not found: {raw_NLCD_map}")
    except FileNotFoundError as e:
        import traceback
        print("".join(traceback.format_exception(type(e), e, e.__traceback__)))
        print('Download manually from https://www.mrlc.gov/data?f%5B0%5D=category%3ALand%20Cover&f%5B1%5D=project_tax_term_term_parents_tax_term_name%3AAnnual%20NLCD&f%5B2%5D=project_tax_term_term_parents_tax_term_name%3AAnnual%20NLCD')
        raise


In [18]:
post_processed_NLCD_crop_filename = crop_nlcd_with_gdalwarp(raw_NLCD_map, year, nlcd_crop_domain, 
                                                output_NLCD_crop_filename='Testing_NLCD_crop.tif')

Cropped NLCD file already exists: /rhome/akumar/tmp/nlcd_processed/Testing_NLCD_crop.tif. Use force_recalculate=True to overwrite.


In [19]:
nlcd_2017 = post_processed_NLCD_crop_filename

with rasterio.open(nlcd_2017) as src:
    lulc_2017 = src.read(1)[::50, ::50]

urban_classes = [21, 22, 23, 24]

lon_min, lon_max, lat_min, lat_max = nlcd_crop_domain['lon_min'], nlcd_crop_domain['lon_max'], nlcd_crop_domain['lat_min'], nlcd_crop_domain['lat_max']

nlcd_longitudes = np.linspace(lon_min, lon_max, lulc_2017.shape[1])
nlcd_latitudes = np.linspace(lat_min, lat_max, lulc_2017.shape[0])

lulc_urban_2017 = np.where((lulc_2017 >= 21) & (lulc_2017 <= 24), lulc_2017, np.nan)

lulc_plot_2017 = np.copy(lulc_urban_2017)
lulc_plot_2017[~np.isin(lulc_plot_2017, urban_classes)] = np.nan

nlcd_lulc_da_2017 = xr.DataArray(
    np.flipud(lulc_plot_2017),
    dims=("lat", "lon"),
    coords={"lat": nlcd_latitudes, "lon": nlcd_longitudes},
    name="Urban_LULC_2017"
)

print(nlcd_lulc_da_2017.mean())

<xarray.DataArray 'Urban_LULC_2017' ()> Size: 8B
array(21.78236375)


In [ ]:

# --- Interpolate NLCD LULC to geog grid for 2017 only ---
nlcd_lulc_da_2017_interp = nlcd_lulc_da_2017.interp(lat=geog_latitudes_1d, lon=geog_longitudes_1d, method='nearest')

# --- Map NLCD urban classes to WRF/GEOG urban classes ---
nlcd_to_geog_urban = {21: 23, 22: 24, 23: 25, 24: 26}

# --- Create updated LU_INDEX array for 2017 ---
geog01_lulc_updated_2017 = np.copy(geog01_lulc)

for nlcd_val, geog_val in nlcd_to_geog_urban.items():
    mask_2017 = nlcd_lulc_da_2017_interp == nlcd_val
    geog01_lulc_updated_2017[mask_2017] = geog_val

# --- Generate LANDUSEF array from updated LU_INDEX for 2017 ---
landuse_2017 = generate_landusef_from_lu_index(geog01_lulc_updated_2017, nclass=40, dtype=np.float32)

# --- Write new geo_em file with updated LU_INDEX and LANDUSEF for 2017 ---
import shutil

outfile_geog01_2017 = infile_geog01.replace('.nc', '_2017.nc')
shutil.copy(infile_geog01, outfile_geog01_2017)

import netCDF4

def update_lu_index_and_landusef_in_netcdf(ncfile, new_lu_index, new_landusef):
    with netCDF4.Dataset(ncfile, mode='r+') as ds:
        # Update LU_INDEX
        lu_index_var = ds.variables['LU_INDEX']
        if new_lu_index.shape != lu_index_var.shape:
            if lu_index_var.shape[0] == 1 and new_lu_index.ndim == 2:
                lu_index_to_write = new_lu_index[np.newaxis, :, :]
            else:
                lu_index_to_write = np.reshape(new_lu_index, lu_index_var.shape)
        else:
            lu_index_to_write = new_lu_index
        lu_index_var[:] = lu_index_to_write

        # Update LANDUSEF
        landusef_var = ds.variables['LANDUSEF']
        if new_landusef.shape != landusef_var.shape:
            if landusef_var.shape[0] == 1 and new_landusef.shape[0] == 1 and new_landusef.shape[1:] == landusef_var.shape[1:]:
                landusef_to_write = new_landusef
            else:
                landusef_to_write = np.reshape(new_landusef, landusef_var.shape)
        else:
            landusef_to_write = new_landusef
        landusef_var[:] = landusef_to_write

update_lu_index_and_landusef_in_netcdf(outfile_geog01_2017, geog01_lulc_updated_2017, landuse_2017)

print(f"Created new geo_em file:\n  {outfile_geog01_2017}")
